<a href="https://colab.research.google.com/github/t-j-fryer/oPool_Optimiser/blob/main/notebooks/oPool_Cloning_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# oPool Optimiser — Google Colab

Generate pooled Golden Gate cloning oligos without installing anything locally. oPool Optimiser codon-optimizes protein libraries, chooses synonymous split sites and compatible overhangs, assigns sub-pools, and adds the primer and assembly sequences needed to order the library.

<img src="https://raw.githubusercontent.com/t-j-fryer/oPool_Optimiser/main/docs/images/opool_computational_workflow.png" alt="Computational oPool workflow" width="100%"/>

**A single oPool is selectively amplified into many defined sub-pools.** A small aliquot of the pooled DNA is added once to a shared PCR master mix and distributed across wells containing sub-pool-specific primers, minimizing consumption and handling of the purchased pool.

<img src="https://raw.githubusercontent.com/t-j-fryer/oPool_Optimiser/main/docs/images/opool_wet_lab_workflow.png" alt="Wet-lab oPool workflow" width="100%"/>

After design, follow the [sub-pool amplification protocol](https://github.com/t-j-fryer/oPool_Optimiser#wet-lab-protocol-sub-pool-specific-opool-amplification), including the shared **3 µL total oPool DNA** setup tested for up to 384 PCRs.

1. In **User settings**, choose either bundled amino-acid dataset or turn bundled data off to upload one amino-acid or optimized-DNA CSV.
2. Choose **Runtime → Run all**.
3. If uploading a file, select it when prompted.
4. Download the result ZIP when the workflow finishes.

A standard CPU runtime is sufficient; a GPU or TPU will not accelerate this workflow. Uploaded inputs and temporary outputs exist in the Colab runtime. They are deleted when the runtime is recycled unless you download the ZIP or enable the optional Google Drive output.


## Automatic setup — do not edit

This downloads the public repository and installs its small Colab dependency set. The source revision used for the run is printed for traceability.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = "https://github.com/t-j-fryer/oPool_Optimiser.git"
REPOSITORY_REF = "main"
PROJECT_ROOT = Path("/content/oPool_Optimiser")

if not (PROJECT_ROOT / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPOSITORY_REF, REPOSITORY_URL, str(PROJECT_ROOT)],
        check=True,
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "-r", str(PROJECT_ROOT / "requirements-colab.txt")],
    check=True,
)
SOURCE_REVISION = subprocess.check_output(
    ["git", "-C", str(PROJECT_ROOT), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
print(f"oPool Optimiser ready (source revision {SOURCE_REVISION}).")


## User settings — edit this cell only

`GENES_PER_SUBPOOL = 0` means automatic packing with no fixed gene-count limit. The repository overhang and primer inventories are used automatically.

### Exploring oligo length

Try the same dataset at **250 nt** and **350 nt**. Shorter oligos leave less room for coding DNA after primers and assembly elements, so genes generally require more fragments and more total oligos; each gene also consumes more unique internal overhangs, which can reduce the number of genes packed into a sub-pool. Longer oligos generally do the reverse. Vendor pricing and synthesis limits may make the shorter option preferable despite the larger oligo count.

Each run uses a new timestamped folder, so it is safe to rerun at another length. Compare **Total order oligos**, **Sub-pools**, and the genes/oligos-per-sub-pool table printed at the end.


In [ ]:
USE_BUNDLED_DATASET = True  # @param {type:"boolean"}
BUNDLED_DATASET = "AAseq_dTF001_dTF016.csv"  # @param ["AAseq_dTF001_dTF016.csv", "fpbase_top500.csv"]
OPOOL_LENGTH = 250  # @param {type:"slider", min:250, max:350, step:10}
VECTOR_OVERHANG_1 = "GCTT"  # @param {type:"string"}
VECTOR_OVERHANG_2 = "AGTG"  # @param {type:"string"}
GENES_PER_SUBPOOL = 0  # @param {type:"integer"}
CODON_SPECIES = "e_coli"  # @param {type:"string"}
STRIP_NTERM_MET = True  # @param {type:"boolean"}
PRIMER_MODE = "combinatorial"  # @param ["combinatorial", "unique_pairs"]
TYPEIIS_RECOGNITION_SITE = "GGTCTC"  # @param {type:"string"}
TYPEIIS_N_BASE = "A"  # @param {type:"string"}
SAVE_TO_GOOGLE_DRIVE = False  # @param {type:"boolean"}
AUTO_DOWNLOAD_RESULTS = True  # @param {type:"boolean"}
RUN_NAME = ""  # @param {type:"string"}


## Input and output preparation — automatic

When bundled data is disabled, **Run all** pauses here for one CSV upload. Google Drive is mounted only when its output option is explicitly enabled.


In [ ]:
from datetime import datetime, timezone
import re

from google.colab import files

BUNDLED_DATASETS = {
    "AAseq_dTF001_dTF016.csv": PROJECT_ROOT / "data" / "AAseq_dTF001_dTF016.csv",
    "fpbase_top500.csv": PROJECT_ROOT / "data" / "fpbase_top500.csv",
}
if USE_BUNDLED_DATASET:
    if BUNDLED_DATASET not in BUNDLED_DATASETS:
        raise ValueError(f"Unknown bundled dataset: {BUNDLED_DATASET}")
    INPUT_FILE = BUNDLED_DATASETS[BUNDLED_DATASET]
else:
    print("Select one amino-acid CSV or optimized-DNA CSV.")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError(f"Upload exactly one CSV file; received {len(uploaded)}.")
    uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
    if Path(uploaded_name).suffix.lower() != ".csv":
        raise ValueError("The uploaded input must be a .csv file.")
    upload_dir = Path("/content/opool_uploads")
    upload_dir.mkdir(parents=True, exist_ok=True)
    INPUT_FILE = upload_dir / Path(uploaded_name).name
    INPUT_FILE.write_bytes(uploaded_bytes)

if not isinstance(GENES_PER_SUBPOOL, int) or GENES_PER_SUBPOOL < 0:
    raise ValueError("GENES_PER_SUBPOOL must be 0 (automatic) or a positive integer.")
GENES_PER_SUBPOOL_VALUE = None if GENES_PER_SUBPOOL == 0 else GENES_PER_SUBPOOL

default_run_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", INPUT_FILE.stem).strip("._") or "opool_run"
RUN_NAME_VALUE = re.sub(r"[^A-Za-z0-9_.-]+", "_", RUN_NAME.strip()).strip("._") if RUN_NAME.strip() else default_run_name
run_timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")

if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = Path("/content/drive/MyDrive/oPool_Optimiser_results") / f"{RUN_NAME_VALUE}_{run_timestamp}"
else:
    OUTPUT_DIR = Path("/content/opool_results") / f"{RUN_NAME_VALUE}_{run_timestamp}"

OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
OVERHANGS_FILE = PROJECT_ROOT / "data" / "overhangs.csv"
PRIMERS_FILE = PROJECT_ROOT / "data" / "orthogonal_oligos.csv"
print(f"Input:  {INPUT_FILE}")
print(f"Output: {OUTPUT_DIR}")


## Run the complete workflow — automatic


In [ ]:
import pandas as pd

from opool_workflow import WorkflowConfig, run_workflow

config = WorkflowConfig(
    input_path=INPUT_FILE,
    output_dir=OUTPUT_DIR,
    run_name=RUN_NAME_VALUE,
    overhangs_path=OVERHANGS_FILE,
    primers_path=PRIMERS_FILE,
    opool_length=OPOOL_LENGTH,
    genes_per_subpool=GENES_PER_SUBPOOL_VALUE,
    short_pool_max_size=None,
    vector_oh1=VECTOR_OVERHANG_1,
    vector_oh2=VECTOR_OVERHANG_2,
    codon_species=CODON_SPECIES,
    strip_nterm_met=STRIP_NTERM_MET,
    primer_mode=PRIMER_MODE,
    typeiis_site=TYPEIIS_RECOGNITION_SITE,
    typeiis_n=TYPEIIS_N_BASE,
    force=False,
)
result = run_workflow(config)

assigned_table = pd.read_csv(result.paths.assigned)
order_table = pd.read_csv(result.paths.fragments)
fragment_columns = [column for column in assigned_table.columns if column.startswith("DNA Fragment")]
assigned_table["Order oligos"] = assigned_table[fragment_columns].apply(
    lambda row: sum(pd.notna(value) and str(value).strip().upper() not in {"", "N/A"} for value in row),
    axis=1,
)
subpool_summary = (
    assigned_table.groupby("Block", as_index=False)
    .agg(Genes=("Sequence Name", "nunique"), **{"Order oligos": ("Order oligos", "sum")})
    .rename(columns={"Block": "Sub-pool"})
)
genes_per_subpool = subpool_summary["Genes"]
summary = pd.DataFrame([{
    "Source revision": SOURCE_REVISION,
    "Optimized genes": result.optimized_genes,
    "Assigned genes": result.assigned_genes,
    "Unassigned genes": result.unassigned_genes,
    "Total order oligos": len(order_table),
    "Sub-pools": result.blocks,
    "Genes/sub-pool (min–median–max)": (
        f"{int(genes_per_subpool.min())}–{genes_per_subpool.median():g}–{int(genes_per_subpool.max())}"
        if not genes_per_subpool.empty else "N/A"
    ),
    "Runtime (seconds)": round(result.runtime_seconds, 3),
}])
display(summary)
display(subpool_summary)
display(pd.DataFrame({"Output file": [str(path) for path in result.output_files]}))


## Package and download results — automatic

A ZIP is created for every run. If automatic download is blocked, open the Files panel on the left and download the printed ZIP path manually.


In [ ]:
import shutil

archive_base = Path("/content") / f"{RUN_NAME_VALUE}_oPool_results_{run_timestamp}"
archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=OUTPUT_DIR))
print(f"Results ZIP: {archive_path}")
if SAVE_TO_GOOGLE_DRIVE:
    print(f"A persistent copy of the uncompressed outputs is in: {OUTPUT_DIR}")
if AUTO_DOWNLOAD_RESULTS:
    files.download(str(archive_path))
